<a href="https://colab.research.google.com/github/Srushanth/RAG/blob/agentic-rag/Agentic-RAG/notebooks/01_llamaindex_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦙 Agentic RAG with LlamaIndex

LlamaIndex makes it incredibly easy to turn query engines into tools for an agent. Here we use the `ReActAgent`.

In [1]:
! pip install --quiet "llama-index-core>=0.14.21" "llama-index-embeddings-huggingface>=0.7.0" "llama-index-llms-google-genai>=0.9.2" "sentence-transformers>=5.4.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 16.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
import os
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import getpass

In [ ]:
os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")

GEMINI_API_KEY: ··········


In [ ]:
from llama_index.core import Settings
from llama_index.core.agent import ReActAgent
from llama_index.core import VectorStoreIndex
from llama_index.core.tools import ToolMetadata
from llama_index.core.tools import QueryEngineTool
from llama_index.core import SimpleDirectoryReader
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.agent.workflow import AgentStream
from llama_index.core.agent.workflow import ToolCallResult
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cuda")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print("Loading Data...")
documents = SimpleDirectoryReader("data").load_data()

Loading Data...


In [ ]:
print('Building Index...')
index = VectorStoreIndex.from_documents(documents, show_progress=True)
query_engine = index.as_query_engine(similarity_top_k=3)

Building Index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
vector_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name='modular_rag_blog_post',
        description='Useful for answering questions about Modular RAG blog post.'
    )
)

In [ ]:
agent = ReActAgent(tools=[vector_tool], llm=Settings.llm, verbose=True)
print(f"Type of agent: {type(agent)}")
print(f"Available attributes/methods on agent: {dir(agent)}")

Type of agent: <class 'llama_index.core.agent.workflow.react_agent.ReActAgent'>
Available attributes/methods on agent: ['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__p

In [ ]:
response = agent.run("What is this document about?")

async for ev in response.stream_events():
    if isinstance(ev, ToolCallResult):
        print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
    if isinstance(ev, AgentStream):
        print(f"{ev.delta}", end="", flush=True)

response = await response

[tick] add: AgentWorkflowStartEvent(user_msg='What is this document about?', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='What is this document about?')])], current_agent_name='Agent')
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='What is this document about?')])], current_agent_name='Agent')
[run_agent_step:0] started from AgentSetup
Observation: The document is about Modular RAG, a new approach to Retrieval Augmented Generation that enhances the capabilities of Large Language Models by integrating various modules for different stages of the RAG pipeline

In [ ]:
print(str(response))

This document is about Modular RAG (Retrieval Augmented Generation). It discusses how Modular RAG improves upon traditional RAG by integrating various modules into the RAG pipeline to enhance the capabilities of Large Language Models. The document covers the limitations of traditional RAG, introduces the concept of Modular RAG, details its key components, benefits, and future implications, emphasizing how it can lead to more accurate, relevant, and efficient LLM-generated responses through dynamic and context-aware retrieval and generation.
